# any-reduce-axis — worked example 1: Column-wise any() to flag active sensors

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `any-reduce-axis`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

`.any(dim=k)` collapses one axis of a boolean tensor with a logical OR. The collapsed axis is the one named by `dim`, so for an `(N, M)` mask, `dim=0` reduces over the N rows and yields one bool per column (shape `(M,)`), while `dim=1` reduces over the M columns and yields one bool per row (shape `(N,)`). Choosing the right `dim` is purely about *which* axis disappears.

## Worked solution

We have a `(timesteps, sensors)` boolean tensor `readings` where `readings[t, s]` is `True` if sensor `s` fired at time `t`. We want a `(sensors,)` vector telling us which sensors ever fired.

**Step 1 — identify the axis to collapse.** "Ever fired" means OR across all timesteps for a fixed sensor. The timestep axis is axis 0. We want it to *disappear*, leaving the sensor axis. So we reduce with `dim=0`.

**Step 2 — apply `.any(dim=0)`.** This ORs down each column. Output shape is `(sensors,)` — exactly the axis we kept. Entry `s` is `True` iff at least one `readings[t, s]` was `True`.

**Step 3 — confirm dtype.** `.any()` always returns a `torch.bool` tensor regardless of `keepdim`, so no casting is needed.

The key mental check: `dim=0` collapses rows (per-column result), `dim=1` collapses columns (per-row result). Picking `dim=0` here keeps the sensor axis, which is what "per sensor" demands.

In [ ]:
def active_sensors(readings):
    # readings: (timesteps, sensors) bool -> (sensors,) bool
    return readings.any(dim=0)

t.manual_seed(0)
readings = t.rand(4, 5) > 0.7  # (timesteps=4, sensors=5) bool
result = active_sensors(readings)
print(readings.int())
print(result, result.shape, result.dtype)